# Can learned order help an asynchronous auditory figure emerge?
### SeqSFG • listening, learning, cue challenges, and a complete research plan

**The proposed first finding:** prior training on an arbitrary temporal order improves extraction
of that figure from a sound mixture, beyond component-frequency familiarity and common practice.
Whether that improvement reflects grouping, attention, or recognition is the next question.

[Open in Colab](https://colab.research.google.com/github/MeysamAmirsardari/SeqSFG_task/blob/main/Notebooks/SeqSFG_research_workbench.ipynb)
· [Source repository](https://github.com/MeysamAmirsardari/SeqSFG_task)

**Start:** choose **Runtime → Run all**, then use the buttons below. A CPU runtime is sufficient;
no GPU or paid service is needed. Setup loads a pinned revision of the repository. Audio only
starts when you press Play. Use headphones at a comfortable, fixed level.

| Stop | What you can do | What it resolves |
|---|---|---|
| 1 | Hear arbitrary order P, its reversal Q, and reshuffling | Is the percept accessible? What changes with asynchrony? |
| 2 | Take a pretest → training → posttest with downloadable responses | Is this a usable learning task? |
| 3 | Run invariants, cue observers, and a planted-confound test | Could another feature support the judgment? |
| 4 | Inspect mechanism assumptions and a power sensitivity analysis | Which claims and sample sizes are defensible? |
| 5 | Export and read the full candidate protocol | What exactly should the next study do? |

This notebook is a **research workbench**, not a validated online experiment. Demonstration trials
do not constitute evidence for learning. The default pilot has known alternative cues. All
simulated effects are labelled assumptions; none are represented as observed human results.

In [ ]:
#@title 0. Load the repository and notebook dependencies
import os, sys, subprocess, importlib.util
from pathlib import Path
import json

REPO_URL = 'https://github.com/MeysamAmirsardari/SeqSFG_task.git'
SOURCE_REF = 'c83afed07cda5d1d85443564194bfcb30e4bed39'
IN_COLAB = importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
if IN_COLAB:
    ROOT = Path('/content') / ('seqsfg-workbench-' + SOURCE_REF[:12])
    if not ROOT.exists():
        subprocess.run(['git','clone','--no-checkout',REPO_URL,str(ROOT)],check=True)
        subprocess.run(['git','-C',str(ROOT),'checkout','--detach',SOURCE_REF],check=True)
    actual = subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
    if actual != SOURCE_REF:
        raise RuntimeError('Existing clone has another revision. Use a fresh Colab runtime.')
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    ROOT = next((p for p in candidates if (p/'seqsfg'/'stimulus.py').exists()), None)
    if ROOT is None:
        raise RuntimeError('Run locally from the repository, or use the Colab link.')

needed = [name for name in ['numpy','scipy','matplotlib','ipywidgets'] if importlib.util.find_spec(name) is None]
if needed:
    subprocess.run([sys.executable,'-m','pip','install','-q',*needed],check=True)
sys.path.insert(0,str(ROOT))
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Audio, Markdown, HTML, clear_output, FileLink
SOURCE_COMMIT = subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
print('Loaded:', ROOT)
print('Source commit:', SOURCE_COMMIT)
print('No microphone, account sign-in, or server-side audio playback is required.')

## 1. The stimulus and the precise manipulation

The current pilot uses 24 channels, seven figure components, seven repetitions, 60 ms tones,
and 3.2 s intervals. Each channel spends the same 22-tone budget in target and foil. The target
repeats one frequency set; the foil redraws the set while preserving grouped element timing.

This notebook adds **explicit arbitrary P/Q orders** without modifying the package. Q reverses P
in time; both use the same frequencies and have the same sequence of frequency-jump magnitudes
in reverse. Neither is an ascending sweep. Across listeners, which order receives training is
counterbalanced. Training assignment never enters test waveform generation.

At zero onset separation P and Q are **exactly the same physical stimulus**. Treat synchrony as one
anchor, not as two independent trained/untrained conditions. The 14 and 23 ms settings are provisional
pilot choices, not a predicted STDP peak. With 60 ms tones, even 23 ms retains about 62% adjacent
overlap: this pilot tests weakened synchrony, not fully nonoverlapping components.

In [ ]:
#@title Stimulus helpers — run once
import hashlib
from dataclasses import replace
from scipy import stats
from seqsfg.config import Config, validate
from seqsfg.stimulus import (
    Interval, Trial, FIGURE, BACKGROUND, PlacementError, sample_figure_set,
    sample_schedule, place_free, build_redrawn, render_interval, check_invariants,
)
from seqsfg import measure

PILOT = Config.from_dict(json.loads((ROOT / 'pilot_config.json').read_text()))
PROTOCOL_VERSION = 'SeqSFG-order-learning-1.0'
# The imported pilot remains unchanged; all prototype manipulations live here.
CFG = PILOT
D = validate(CFG)
SR = CFG.sample_rate


def delay_orders(n=7, seed=104):
    """Delay rank per frequency channel; Q is the temporal reversal of arbitrary P."""
    rng = np.random.default_rng(seed)
    while True:
        p = rng.permutation(n)
        if not (np.array_equal(p, np.arange(n)) or np.array_equal(p, np.arange(n)[::-1])):
            return {'P': p, 'Q': n - 1 - p}


ORDERS = delay_orders(CFG.n_components)


def prototype_trial(seed, step_ms, pattern_id='P', trial_order='fixed', cfg=CFG):
    """Arbitrary fixed order or within-trial reshuffle, independent of exposure label.

    Uses repo scheduling, fixed per-channel budgets, refractory placement, foil
    construction and rendering. Does not patch the installed package. No training
    assignment enters stimulus generation: a seed/order/step defines identical
    test acoustics under either exposure assignment.
    """
    d = validate(cfg)
    if pattern_id not in ('P', 'Q') or trial_order not in ('fixed', 'reshuffled'):
        raise ValueError('Choose P/Q and fixed/reshuffled.')
    if step_ms < 0 or (cfg.n_components-1)*step_ms + cfg.tone_dur_ms > cfg.iei_min_ms:
        raise ValueError('This onset step does not fit the element interval.')
    step = cfg.ms_to_grid(float(step_ms))
    n, k, dur = cfg.n_components, cfg.n_elements, d.tone_dur_grid
    if cfg.figure_repeats != 1:
        raise ValueError('This prototype supports figure_repeats=1 only.')
    orders = delay_orders(n)
    anchor = cfg.figure_anchor_seed if cfg.figure_anchor_seed is not None else int(seed)
    channels = sample_figure_set(np.random.default_rng([anchor, 0xF16]),
                                d.n_channels, n, cfg.figure_min_spacing_channels)
    for attempt in range(50):
        rng = np.random.default_rng([int(seed), attempt, 0xD12])
        times = sample_schedule(rng, cfg)
        order_rng = np.random.default_rng([int(seed), attempt, 0x0D3])
        patterns = ([orders[pattern_id].copy() for _ in range(k)] if trial_order == 'fixed'
                    else [order_rng.permutation(n) for _ in range(k)])
        f_on = np.array([times[e] + patterns[e][j]*step for e in range(k) for j in range(n)])
        f_ch = np.tile(channels, k)
        f_el, f_co = np.repeat(np.arange(k), n), np.tile(np.arange(n), k)
        try:
            bg_on, bg_ch = [], []
            for ch in range(d.n_channels):
                fixed = f_on[f_ch == ch]
                extra = place_free(rng, fixed, cfg.n_grid-dur+1, dur,
                                   cfg.tones_per_channel-len(fixed))
                bg_on.extend(extra.tolist()); bg_ch.extend([ch]*len(extra))
            nb, nf = len(bg_on), len(f_on)
            a = Interval('recurring', 'scrambled', float(step_ms),
                         np.r_[f_on, np.asarray(bg_on, int)], np.r_[f_ch, np.asarray(bg_ch, int)],
                         rng.uniform(0, 2*np.pi, nf+nb),
                         np.r_[np.full(nf, FIGURE), np.full(nb, BACKGROUND)],
                         np.r_[f_el, np.full(nb, -1)], np.r_[f_co, np.full(nb, -1)],
                         times, [channels.copy() for _ in range(k)], patterns, channels)
            b = build_redrawn(rng, cfg, d, a)
            return Trial(int(seed), 'scrambled', float(step_ms), a, b, attempt)
        except PlacementError:
            continue
    raise PlacementError('Prototype failed after 50 attempts; revise parameters, not labels.')


def subset_interval(iv, kind):
    out = iv.copy()
    mask = iv.kind == kind
    for name in ('onset', 'channel', 'phase', 'kind', 'element', 'component'):
        setattr(out, name, getattr(iv, name)[mask])
    return out


def render_mixture(iv, background_db=0.0, cfg=CFG):
    if background_db == 0:
        return render_interval(cfg, iv)
    figure = render_interval(cfg, subset_interval(iv, FIGURE))
    background = render_interval(cfg, subset_interval(iv, BACKGROUND))
    return figure + 10**(float(background_db)/20) * background


def provenance(cfg=CFG):
    import platform, scipy, matplotlib
    source_files = [ROOT/'seqsfg'/name for name in ('config.py', 'stimulus.py', 'measure.py')]
    return dict(protocol=PROTOCOL_VERSION, source_commit=SOURCE_COMMIT,
                core_source_sha256=hashlib.sha256(b''.join(p.read_bytes() for p in source_files)).hexdigest(),
                config=cfg.to_dict(), config_hash=cfg.hash(), python=platform.python_version(),
                numpy=np.__version__, scipy=scipy.__version__, matplotlib=matplotlib.__version__,
                delay_orders={name: value.tolist() for name,value in delay_orders(cfg.n_components).items()},
                mode='exploratory browser demonstration; not a validated research session')


def trial_row(seed, step, pattern_id, target_position, phase='test', stage='pre',
              trained_order='P', background_db=0.0, cfg=CFG):
    tr = prototype_trial(seed, step, pattern_id, cfg=cfg)
    a, b = (render_mixture(iv, background_db, cfg) for iv in (tr.recurring,tr.other))
    x1, x2 = (a,b) if target_position == 1 else (b,a)
    return dict(trial_id=f'{stage}-{seed}', audio_1=x1, audio_2=x2,
                correct_interval=int(target_position), phase=phase, stage=stage,
                condition=f'{stage}: '+('trained' if pattern_id==trained_order else 'untrained'),
                pattern_id=pattern_id, trained_order=trained_order, step_ms=float(step), seed=int(seed),
                background_db=float(background_db), rebuilds=tr.n_rebuilds,
                audio_sha256=hashlib.sha256(x1.tobytes()+x2.tobytes()).hexdigest(),
                instruction=('Training — feedback follows each answer. ' if phase=='practice' else
                             f'{stage.capitalize()} check — no answer feedback. ')+
                            'Which interval contains a group that keeps returning at the same pitches?')


def mini_session(trained_order='P', step=14.0, n_per_order=2, exposure_trials=4, seed=6501):
    """Pre/test banks & balancing invariant to training assignment; exposure changes."""
    if n_per_order % 2 or exposure_trials % 2:
        raise ValueError('Even counts are required to balance interval position.')
    if step <= 0:
        raise ValueError('Train at a nonzero lag: P and Q are identical at synchrony.')
    rows=[]
    for stage, offset in [('pre',0),('training',10000),('post',20000)]:
        block=[]
        patterns = [trained_order] if stage=='training' else ['P','Q']
        count = exposure_trials if stage=='training' else n_per_order
        for pi, pat in enumerate(patterns):
            for j in range(count):
                bg = ([-18.0,-12.0,-6.0,0.0][min(3, j*4//count)] if stage=='training' else 0.0)
                block.append(trial_row(seed+offset+pi*100+j,step,pat,1+j%2,
                                       'practice' if stage=='training' else 'test',stage,
                                       trained_order,bg))
        if stage!='training':
            np.random.default_rng(seed+offset+99).shuffle(block)
        rows.extend(block)
    return rows


def construction_checks():
    """Meaningful invariants, distinct from the cue audit or human audibility."""
    for seed in [91,92,93]:
        for step in [0.0,14.0,23.0]:
            for pat in ['P','Q']:
                tr=prototype_trial(seed,step,pat)
                inv=check_invariants(CFG,tr)
                for key in ['same_n_tones','same_channel_counts','budget_exact','no_same_channel_overlap']:
                    assert inv[key], (seed,step,pat,key)
                for iv in [tr.recurring,tr.other]:
                    assert np.max(iv.onset)+D.tone_dur_grid <= CFG.n_grid
                    audio=render_interval(CFG,iv)
                    assert np.isfinite(audio).all() and np.max(np.abs(audio)) < 1
    p,q = (prototype_trial(73,0.0,pat) for pat in ['P','Q'])
    for side in ['recurring','other']:
        assert np.array_equal(render_interval(CFG,getattr(p,side)),render_interval(CFG,getattr(q,side)))
    a=trial_row(93,14.0,'P',1,trained_order='P')
    b=trial_row(93,14.0,'P',1,trained_order='Q')
    assert a['audio_sha256']==b['audio_sha256']
    assert not np.array_equal(prototype_trial(93,14,'P').recurring.onset,
                              prototype_trial(93,14,'Q').recurring.onset)
    print('PASS: budgets, refractory spacing, timing bounds, finite unclipped audio,')
    print('      physical P/Q identity at 0 ms, distinct orders at 14 ms, and exposure-independent test audio.')
    print('These checks do not establish audibility or the absence of alternative cues.')


print(f'Loaded pilot: {D.n_channels} channels, {CFG.tone_dur_ms:g} ms tones, {CFG.n_elements} elements.')
print('P and Q delay ranks:', {k:v.tolist() for k,v in ORDERS.items()})

In [ ]:
#@title Construction tests — actual audio and schedules, not human performance
construction_checks()

In [ ]:
#@title Listening explorer — choose settings, then Build preview
listen_step=widgets.SelectionSlider(options=[0,5,9,14,18,23],value=14,description='Step (ms)')
listen_bg=widgets.SelectionSlider(options=[-24,-18,-12,-6,0],value=-12,description='Cloud (dB)')
listen_pat=widgets.Dropdown(options=['P','Q'],value='P',description='Order')
listen_mode=widgets.Dropdown(options=['fixed','reshuffled'],description='Repetition')
listen_seed=widgets.IntText(value=404,description='Seed')
listen_button=widgets.Button(description='Build preview',button_style='primary')
listen_out=widgets.Output()

def listening_preview(step=14,background_db=-12,pattern_id='P',trial_order='fixed',seed=404):
    tr=prototype_trial(seed,step,pattern_id,trial_order)
    fig,axes=plt.subplots(1,2,figsize=(12,3.4),sharey=True)
    for ax,iv,title in zip(axes,[tr.recurring,tr.other],['Target: same frequency set returns','Foil: sets change']):
        bg=iv.kind==BACKGROUND; fg=iv.kind==FIGURE
        ax.scatter(iv.onset[bg],iv.channel[bg],s=3,color='#a9b3bd',alpha=.5)
        ax.scatter(iv.onset[fg],iv.channel[fg],s=16,color='#007d8a')
        ax.set(xlabel='Time (ms)',title=title,xlim=(0,CFG.interval_dur_ms))
    axes[0].set_ylabel('Frequency channel')
    plt.tight_layout();display(fig);plt.close(fig)
    for label,signal in [
        ('Target figure alone — teaching aid',render_interval(CFG,subset_interval(tr.recurring,FIGURE))),
        ('Target mixture',render_mixture(tr.recurring,background_db)),
        ('Foil mixture',render_mixture(tr.other,background_db))]:
        display(Markdown('**'+label+'**'))
        display(Audio(signal,rate=SR,normalize=False))
    print('Identical physical gain; no per-file loudness normalization.')
    if background_db<0:
        print('The attenuated background teaches the percept; its cues differ from the audited test stimulus.')

def on_listen(_):
    listen_button.disabled=True
    with listen_out:
        clear_output(wait=True)
        try: listening_preview(listen_step.value,listen_bg.value,listen_pat.value,listen_mode.value,listen_seed.value)
        except Exception as e: print(type(e).__name__+': '+str(e))
    listen_button.disabled=False
listen_button.on_click(on_listen)
display(widgets.VBox([widgets.HBox([listen_step,listen_bg]),widgets.HBox([listen_pat,listen_mode,listen_seed]),listen_button,listen_out]))

## 2. Try the learning task

The question throughout is **“Which interval contains a group that keeps returning at the same
pitches?”** Each trial plays interval 1 and interval 2 in sequence. Answers unlock after playback.
Training trials give feedback and progressively raise the background. Pre/post trials give no
correctness feedback until the panel ends.

The default mini-session has **4 pretest + 4 training + 4 posttest trials**. Both test orders contain
the same frequencies. This is far too few trials for an effect estimate, and seeing these labels
means your own run is not blinded. It is a usability test for an instructed-learning paradigm.
For a repeat run, change the seed; repeated self-testing creates additional exposure.

Click **Build session**, then **Play** inside the resulting panel. Keep that panel open until you
download its CSV; rerunning or clearing the output discards unsaved responses. CSVs remain in your
browser unless you choose to download/upload them. Browser response times and sound level are
uncalibrated; use accuracy as the demonstration outcome.

In [ ]:
#@title Browser trial interface — run once
"""Colab-compatible browser trial panel; no callbacks, network requests, or autoplay.

Usage:
    display(browser_trial_panel(rows, sample_rate=cfg.sample_rate, provenance={...}))

Each row requires:
    trial_id: unique string/int
    audio_1, audio_2: nonempty one-dimensional mono float arrays in [-1, 1]
    correct_interval: integer 1 or 2
    phase: 'practice' (immediate feedback) or 'test' (no trial feedback)
Optional:
    instruction: participant-facing task instruction (no answer/condition leakage)
    condition: a condition identifier for final descriptive summaries
All remaining row fields are exported as metadata. Arrays are encoded as PCM16 WAV
without normalization; use the same physical rendering gain across all conditions.
Provenance should contain the repo commit, configuration/hash, seed and task version.
The returned HTML stores responses only in its browser output until CSV download.
Rerunning a panel creates a new session; export the old panel before clearing it.
"""
import base64
import io
import json
import uuid
import wave
from datetime import datetime, timezone

import numpy as np
from IPython.display import HTML


def browser_trial_panel(trial_rows, sample_rate, provenance):
    """Return self-contained HTML for a two-interval browser demonstration.

    Answers unlock only after both intervals have finished. Replaying is allowed
    before answering and logged. Test feedback is withheld until the end-of-block
    descriptive summary. RT is a browser estimate, not calibrated reaction time.
    """
    sample_rate = int(sample_rate)
    if sample_rate < 8000 or sample_rate > 192000:
        raise ValueError('sample_rate must be between 8,000 and 192,000 Hz')

    def json_default(value):
        if isinstance(value, np.ndarray):
            return value.tolist()
        if isinstance(value, np.generic):
            return value.item()
        raise TypeError(f'Metadata must be JSON serializable: {type(value).__name__}')

    def encode_audio(values):
        arr = np.asarray(values, dtype=float)
        if arr.ndim != 1 or arr.size == 0 or not np.isfinite(arr).all():
            raise ValueError('Audio must be a finite, nonempty mono array')
        if np.max(np.abs(arr)) > 1.0:
            raise ValueError('Audio exceeds [-1, 1]; set one common rendering gain before building the panel')
        stream = io.BytesIO()
        with wave.open(stream, 'wb') as wav:
            wav.setnchannels(1)
            wav.setsampwidth(2)
            wav.setframerate(sample_rate)
            wav.writeframes(np.rint(arr * 32767).astype('<i2').tobytes())
        return base64.b64encode(stream.getvalue()).decode('ascii')

    packed = []
    identifiers = set()
    for raw in trial_rows:
        row = dict(raw)
        trial_id = str(row['trial_id'])
        if trial_id in identifiers:
            raise ValueError(f'Duplicate trial_id: {trial_id}')
        identifiers.add(trial_id)
        if row['correct_interval'] not in (1, 2):
            raise ValueError('correct_interval must be 1 or 2')
        if row.get('phase') not in ('practice', 'test'):
            raise ValueError("phase must be 'practice' or 'test'")
        audio1 = encode_audio(row.pop('audio_1'))
        audio2 = encode_audio(row.pop('audio_2'))
        packed.append({'meta': row, 'audio': [audio1, audio2]})
    if not packed:
        raise ValueError('At least one trial is required')

    payload = {'trials': packed, 'sample_rate': sample_rate,
               'provenance': provenance,
               'created_utc': datetime.now(timezone.utc).isoformat()}
    data = json.dumps(payload, default=json_default, allow_nan=False)
    # Prevent metadata from ending the script element or injecting markup.
    data = data.replace('<', '\\u003c').replace('>', '\\u003e').replace('&', '\\u0026')
    data = data.replace('\u2028', '\\u2028').replace('\u2029', '\\u2029')
    panel_id = 'seqsfg_' + uuid.uuid4().hex
    template = r'''<div id="__PANEL_ID__" class="seqsfg-panel"></div>
<style>
#__PANEL_ID__ {max-width:820px;padding:22px;border:1px solid #bfd0df;border-radius:14px;background:#f8fbfe;color:#172d40;font:15px/1.5 system-ui,sans-serif;}
#__PANEL_ID__ h3 {margin:0 0 8px;font-size:22px;}
#__PANEL_ID__ p {margin:8px 0;}
#__PANEL_ID__ button {font:inherit;border:1px solid #426a8b;border-radius:7px;padding:10px 16px;margin:5px 8px 5px 0;background:white;color:#153d5b;cursor:pointer;}
#__PANEL_ID__ button:disabled {opacity:.45;cursor:default;}
#__PANEL_ID__ .primary {background:#155f91;color:white;}
#__PANEL_ID__ .status {min-height:48px;padding:12px;background:#e6f0f8;border-radius:8px;}
#__PANEL_ID__ .small {font-size:12px;color:#455d70;}
#__PANEL_ID__ table {border-collapse:collapse;margin-top:14px;width:100%;font-size:13px;}
#__PANEL_ID__ th, #__PANEL_ID__ td {text-align:left;border-bottom:1px solid #ccd7df;padding:7px;}
#__PANEL_ID__ progress {width:100%;height:12px;}
</style>
<script>
(() => {
  'use strict';
  const data = __PAYLOAD__;
  const root = document.getElementById('__PANEL_ID__');
  if (!root || root.dataset.initialized) return;
  root.dataset.initialized = 'true';
  const el = (tag, text, cls) => {
    const node = document.createElement(tag);
    if (text !== undefined) node.textContent = String(text);
    if (cls) node.className = cls;
    return node;
  };
  const title = el('h3', 'Listen and choose'); root.appendChild(title);
  root.appendChild(el('p', 'Which interval contains the recurring figure? Listen to both intervals, then choose 1 or 2. Start at a comfortable volume.'));
  const instruction = el('p'); root.appendChild(instruction);
  const progressText = el('p'); root.appendChild(progressText);
  const progress = el('progress'); progress.max = data.trials.length; progress.value = 0; root.appendChild(progress);
  const gainWrap = el('p');
  const gainLabel = el('label', 'Playback level: ');
  const gainSlider = el('input'); gainSlider.type = 'range'; gainSlider.min = '0.01'; gainSlider.max = '1'; gainSlider.step = '0.01'; gainSlider.value = '0.20'; gainSlider.setAttribute('aria-label', 'Playback level');
  const gainText = el('span', ' 20%'); gainLabel.append(gainSlider, gainText); gainWrap.appendChild(gainLabel); root.appendChild(gainWrap);
  const controls = el('div');
  const playButton = el('button', 'Play intervals 1 → 2', 'primary');
  const answer1 = el('button', 'Choose interval 1');
  const answer2 = el('button', 'Choose interval 2');
  const nextButton = el('button', 'Next trial');
  controls.append(playButton, answer1, answer2, nextButton); root.appendChild(controls);
  const status = el('p', 'Click Play to begin.', 'status'); status.setAttribute('role', 'status'); status.setAttribute('aria-live', 'polite'); root.appendChild(status);
  const score = el('p'); root.appendChild(score);
  const exportButton = el('button', 'Download responses CSV'); exportButton.disabled = true; root.appendChild(exportButton);
  root.appendChild(el('p', 'Demonstration only. Timing and sound level are not calibrated. RT is estimated by this browser from the last playback ending. Responses stay in this output until you download them; rerunning or clearing the output does not save them.', 'small'));
  const summary = el('div'); root.appendChild(summary);

  let index = 0, context = null, outputGain = null, decoded = null;
  let playing = false, answered = false, plays = 0, firstEnd = null, lastEnd = null;
  let firstStart = null, presentations = [], activeSources = [], intervalTimer = null;
  let interruptedPlays = 0, hiddenDuringTrial = false, playbackToken = 0;
  const responses = [];
  const sessionId = (globalThis.crypto && crypto.randomUUID) ? crypto.randomUUID() : ('browser-' + Date.now() + '-' + Math.random().toString(16).slice(2));
  const sessionStart = new Date().toISOString();

  function showTrial() {
    const row = data.trials[index].meta;
    answered = false; playing = false; plays = 0; decoded = null;
    firstEnd = null; lastEnd = null; firstStart = null; presentations = [];
    interruptedPlays = 0; hiddenDuringTrial = false;
    instruction.textContent = row.instruction || '';
    progressText.textContent = 'Trial ' + (index + 1) + ' of ' + data.trials.length + (row.phase === 'practice' ? ' · practice with feedback' : ' · test, feedback at end');
    progress.value = responses.length;
    playButton.textContent = 'Play intervals 1 → 2'; playButton.disabled = false;
    answer1.disabled = true; answer2.disabled = true; nextButton.disabled = true;
    gainSlider.disabled = false; status.textContent = 'Click Play when ready.';
    score.textContent = '';
  }

  function decodeWav(b64) {
    const raw = atob(b64), bytes = new Uint8Array(raw.length);
    for (let k = 0; k < raw.length; k++) bytes[k] = raw.charCodeAt(k);
    return context.decodeAudioData(bytes.buffer);
  }

  async function playPair() {
    if (playing || answered || index >= data.trials.length) return;
    playing = true;
    const ownToken = ++playbackToken;
    playButton.disabled = true; answer1.disabled = true; answer2.disabled = true; gainSlider.disabled = true;
    status.textContent = 'Preparing audio…';
    let thisPresentation = null;
    try {
      if (!context) {
        const AC = window.AudioContext || window.webkitAudioContext;
        if (!AC) throw new Error('Web Audio is unavailable in this browser.');
        context = new AC(); outputGain = context.createGain(); outputGain.connect(context.destination);
      }
      await context.resume();
      if (context.state !== 'running') throw new Error('Audio could not start. Keep this output visible and click Play again.');
      if (!decoded) decoded = await Promise.all(data.trials[index].audio.map(decodeWav));
      if (ownToken !== playbackToken || !playing) return;
      outputGain.gain.value = Number(gainSlider.value);
      const start = context.currentTime + 0.08;
      const second = start + decoded[0].duration + 0.65;
      thisPresentation = {presentation_number: presentations.length + 1, gain: Number(gainSlider.value), started_utc: new Date().toISOString(), browser_start_ms: performance.now(), context_start_s: start, interval2_start_s: second, context_end_s: second + decoded[1].duration, completed: false};
      presentations.push(thisPresentation);
      if (firstStart === null) firstStart = thisPresentation.browser_start_ms;
      status.textContent = 'Playing interval 1…';
      intervalTimer = setTimeout(() => { if (playing) status.textContent = 'Playing interval 2…'; }, Math.max(0, (second - context.currentTime) * 1000));
      const source1 = context.createBufferSource(), source2 = context.createBufferSource();
      source1.buffer = decoded[0]; source2.buffer = decoded[1];
      source1.connect(outputGain); source2.connect(outputGain); activeSources = [source1, source2];
      source2.onended = () => {
        if (!playing) return;
        clearTimeout(intervalTimer); activeSources = []; playing = false; plays += 1;
        lastEnd = performance.now(); if (firstEnd === null) firstEnd = lastEnd;
        thisPresentation.completed = true; thisPresentation.browser_ended_ms = lastEnd;
        playButton.textContent = 'Replay both intervals'; playButton.disabled = false;
        answer1.disabled = false; answer2.disabled = false; gainSlider.disabled = false;
        status.textContent = 'Choose interval 1 or 2. You may replay both before answering.';
      };
      source1.start(start); source2.start(second);
    } catch (error) {
      playing = false; clearTimeout(intervalTimer);
      for (const source of activeSources) { source.onended = null; try {source.stop();} catch (_) {} }
      activeSources = [];
      if (thisPresentation) thisPresentation.error = String(error.message || error);
      status.textContent = 'Audio could not play: ' + String(error.message || error) + ' Click Play to retry.';
      playButton.disabled = false; gainSlider.disabled = false;
      answer1.disabled = lastEnd === null; answer2.disabled = lastEnd === null;
    }
  }

  function answer(choice) {
    if (playing || answered || lastEnd === null) return;
    answered = true;
    const now = performance.now(), row = data.trials[index].meta;
    const correct = Number(choice === Number(row.correct_interval));
    const record = {};
    // Prefixes keep user metadata and measured response fields from overwriting each other.
    Object.keys(row).forEach(key => { record['meta_' + key] = row[key]; });
    Object.assign(record, {session_id: sessionId, session_started_utc: sessionStart,
      trial_number: index + 1, choice: choice, correct: correct,
      response_utc: new Date().toISOString(), rt_browser_ms_since_last_pair_end: Math.round(now-lastEnd),
      browser_ms_since_first_pair_end: Math.round(now-firstEnd), browser_trial_elapsed_ms: Math.round(now-firstStart),
      completed_pair_plays: plays, replay_count: Math.max(0, plays-1),
      interrupted_playbacks: interruptedPlays, document_hidden_during_trial: hiddenDuringTrial,
      playback_gain_at_response: Number(gainSlider.value), presentation_log: presentations,
      wav_sample_rate: data.sample_rate, audio_context_sample_rate: context.sampleRate,
      browser_user_agent: navigator.userAgent, panel_created_utc: data.created_utc,
      provenance_json: data.provenance,
      timing_note: 'Approximate performance.now timestamp after Web Audio onended; uncalibrated output latency; RT restarts after replay.'});
    responses.push(record); progress.value = responses.length;
    answer1.disabled = true; answer2.disabled = true; playButton.disabled = true; gainSlider.disabled = true;
    nextButton.disabled = false; exportButton.disabled = false;
    if (row.phase === 'practice') {
      status.textContent = correct ? 'Correct — interval ' + row.correct_interval + '.' : 'The recurring figure was in interval ' + row.correct_interval + '.';
      const practice = responses.filter(r => r.meta_phase === 'practice');
      score.textContent = 'Practice: ' + practice.reduce((a, r) => a + r.correct, 0) + ' / ' + practice.length + ' correct.';
    } else {
      status.textContent = 'Response saved. Continue when ready.'; score.textContent = '';
    }
    nextButton.textContent = index + 1 === data.trials.length ? 'Finish and show summary' : 'Next trial';
  }

  function finish() {
    controls.style.display = 'none'; gainWrap.style.display = 'none'; instruction.textContent = '';
    progressText.textContent = 'Completed ' + responses.length + ' trials.';
    status.textContent = 'Block complete. Download your responses before clearing this output.';
    score.textContent = '';
    summary.appendChild(el('h3', 'Descriptive results'));
    const table = el('table'), head = el('tr');
    ['Phase', 'Condition', 'n', 'Proportion correct'].forEach(label => head.appendChild(el('th', label)));
    const thead = el('thead'); thead.appendChild(head); table.appendChild(thead);
    const tbody = el('tbody'), groups = new Map();
    responses.forEach(r => {
      const condition = r.meta_condition === undefined ? 'all trials' : (typeof r.meta_condition === 'object' ? JSON.stringify(r.meta_condition) : String(r.meta_condition));
      const key = JSON.stringify([r.meta_phase, condition]);
      if (!groups.has(key)) groups.set(key, {phase: r.meta_phase, condition, n: 0, correct: 0});
      const g = groups.get(key); g.n++; g.correct += r.correct;
    });
    groups.forEach(g => {const tr = el('tr'); [g.phase, g.condition, g.n, (g.correct/g.n).toFixed(3)].forEach(v => tr.appendChild(el('td', v))); tbody.appendChild(tr);});
    table.appendChild(tbody); summary.appendChild(table);
    summary.appendChild(el('p', 'Chance is 0.50. These few trials illustrate the task; they are not a formal experiment and cannot establish learning, binding, or a mechanism. Replays, device settings and uncontrolled listening conditions limit interpretation.', 'small'));
  }

  function csvValue(value) {
    let text = value === null || value === undefined ? '' : (typeof value === 'object' ? JSON.stringify(value) : String(value));
    // Quoting protects CSV structure; an apostrophe protects spreadsheet formula-like text.
    if (typeof value !== 'number' && /^[\s]*[=+@-]/.test(text)) text = "'" + text;
    return '"' + text.replace(/"/g, '""') + '"';
  }
  function download() {
    if (!responses.length) return;
    const keys = [...new Set(responses.flatMap(row => Object.keys(row)))];
    const lines = [keys.map(csvValue).join(','), ...responses.map(row => keys.map(key => csvValue(row[key])).join(','))];
    const blob = new Blob(['\uFEFF' + lines.join('\r\n')], {type: 'text/csv;charset=utf-8;'});
    const url = URL.createObjectURL(blob), link = el('a');
    link.href = url; link.download = 'SeqSFG_browser_demo_' + sessionId + '.csv';
    root.appendChild(link); link.click(); link.remove(); setTimeout(() => URL.revokeObjectURL(url), 15000);
  }
  document.addEventListener('visibilitychange', () => {
    if (!document.hidden || answered || index >= data.trials.length) return;
    hiddenDuringTrial = true;
    if (playing) {
      interruptedPlays++; playbackToken++; playing = false; clearTimeout(intervalTimer);
      for (const source of activeSources) {source.onended = null; try {source.stop();} catch (_) {} }
      activeSources = [];
      status.textContent = 'Playback stopped because this page was hidden. Replay both intervals before answering.';
      lastEnd = null; playButton.disabled = false; answer1.disabled = true; answer2.disabled = true; gainSlider.disabled = false;
    }
  });
  gainSlider.addEventListener('input', () => {gainText.textContent = ' ' + Math.round(Number(gainSlider.value)*100) + '%';});
  playButton.addEventListener('click', playPair);
  answer1.addEventListener('click', () => answer(1)); answer2.addEventListener('click', () => answer(2));
  nextButton.addEventListener('click', () => {if (!answered) return; index++; if (index === data.trials.length) finish(); else showTrial();});
  exportButton.addEventListener('click', download);
  showTrial();
})();
</script>'''
    return HTML(template.replace('__PANEL_ID__', panel_id).replace('__PAYLOAD__', data))

In [ ]:
#@title Build an interactive pretest → training → posttest
session_order=widgets.Dropdown(options=['P','Q'],description='Train order')
session_step=widgets.Dropdown(options=[14,23],value=14,description='Step (ms)')
session_n=widgets.Dropdown(options=[2,4,6],value=2,description='Tests/order')
session_exposure=widgets.Dropdown(options=[4,8,12],value=4,description='Training')
session_seed=widgets.IntText(value=6501,description='Seed')
session_button=widgets.Button(description='Build session',button_style='primary')
session_out=widgets.Output()

def on_session(_):
    session_button.disabled=True
    with session_out:
        clear_output(wait=True)
        try:
            rows=mini_session(session_order.value,session_step.value,session_n.value,session_exposure.value,session_seed.value)
            info=provenance();info.update(session_seed=session_seed.value,trained_order=session_order.value)
            display(browser_trial_panel(rows,SR,info))
        except Exception as e: print(type(e).__name__+': '+str(e))
    session_button.disabled=False
session_button.on_click(on_session)
display(widgets.VBox([widgets.HBox([session_order,session_step]),widgets.HBox([session_n,session_exposure,session_seed]),session_button,session_out]))

### Read your result cautiously

The relevant contrast is **(post trained − pre trained) − (post untrained − pre untrained)**.
Improvement on both orders can be ordinary practice; improvement on the trained order alone can
still involve recognition or selective attention. A null result from this tiny block tells us
little. A consistent-versus-reshuffled listening difference is also not a cross-trial learning effect.

The optional upload below summarizes only this notebook's browser CSVs. It never runs a
significance test on a single demonstration session.

In [ ]:
#@title Optional: inspect a downloaded mini-session CSV
import csv, io
upload=widgets.FileUpload(accept='.csv',multiple=False)
upload_out=widgets.Output()
def summarize_demo_csv(text):
    rows=list(csv.DictReader(io.StringIO(text)))
    required=['meta_stage','meta_pattern_id','meta_trained_order','correct']
    if not rows or any(k not in rows[0] for k in required):
        raise ValueError('Upload a CSV exported by this notebook trial panel.')
    means={}
    for stage in ['pre','post']:
        for label in ['trained','untrained']:
            selected=[r for r in rows if r['meta_stage']==stage and
                      ('trained' if r['meta_pattern_id']==r['meta_trained_order'] else 'untrained')==label]
            if selected:
                correct=sum(str(r['correct']).lower() in ['1','true'] for r in selected)
                means[(stage,label)]=correct/len(selected)
                print(f'{stage:4s} {label:9s}: {correct}/{len(selected)} = {correct/len(selected):.2f}')
    if len(means)==4:
        delta=(means['post','trained']-means['pre','trained'])-(means['post','untrained']-means['pre','untrained'])
        print(f'Descriptive change contrast: {delta:+.3f}; no inferential claim from this mini-session.')
    else: print('Incomplete pre/post cells: do not compute the change contrast.')
    return means
def on_upload(change):
    if not upload.value: return
    item=next(iter(upload.value.values())) if isinstance(upload.value,dict) else upload.value[0]
    with upload_out:
        clear_output(wait=True)
        try: summarize_demo_csv(bytes(item['content']).decode('utf-8-sig'))
        except Exception as e: print(type(e).__name__+': '+str(e))
upload.observe(on_upload,names='value')
display(widgets.VBox([upload,upload_out]))

## 3. Challenge the stimulus before interpreting performance

The saved pilot audit is **not clean**: its primary single-channel observer reported d′ = 0.358
(95% CI 0.13–0.58; corrected p = .012); the global cue test reported p = .003. These are stored
simulation outputs from this configuration, not human data or results generated by this notebook.

A development run of this arbitrary-order prototype with **256 pairs per step** also exposed a
regularity cue (about 71% fixed-rule accuracy at 14 ms). Re-run the audit rather than treating
that number as universal. The counterbalanced learning comparison can still test an
order-specific extraction benefit, but claims that success requires binding need stronger controls.

The live audit below tests four prespecified scores: mean and maximum per-channel onset-pair
regularity, broadband envelope variation, and overall RMS. It audits **target versus foil**,
not the causal effect of assigning P versus Q to training. That causal contrast also requires
exposure-trained observers and counterbalanced stimulus banks in the full study.

An observer detecting a cue does not establish that listeners use it. A nonsignificant result
does not establish cue absence. The audit therefore reports uncertainty and whether **both score
directions** can be bounded below a selected accuracy tolerance, with simultaneous intervals.
Small samples can yield **inconclusive** bounds; the default uses 256 pairs per step. Schedule observers have exact
per-channel onset access; this is an idealized diagnostic, not a fitted auditory model.

Use **Plant +5 dB target cue** to verify that this audit can detect a known defect. That gain is
applied to audit measurements only; the listening task remains unchanged. The live check is a
subset of the repository battery; it does not replace the full spectral/occupancy audit.

In [ ]:
#@title Check that the stored pilot report belongs to the loaded configuration
report_path=ROOT/'verification'/'pilot_battery_report.txt'
if report_path.exists():
    report=report_path.read_text()
    print('Loaded configuration hash:',CFG.hash())
    print('Stored report matches:', ('config hash          '+CFG.hash()) in report)
    lines=report.splitlines()
    start=next((i for i,s in enumerate(lines) if "--- ladder 'rising'" in s),None)
    if start is not None: print('\n'.join(lines[start:start+9]))
else:
    print('No saved pilot report in this checkout. Use the live audit below.')

In [ ]:
#@title Cue audit functions — run once
"""Notebook helpers for a fixed-observer, paired SeqSFG cue audit.

These are stimulus diagnostics, not a fitted human model. Detection establishes
an available cue for the specified observer, not human use of that cue. A bound
covers only these observers, these conditions, and this stimulus generator.
"""
import numpy as np
from scipy.stats import beta as _audit_beta, norm as _audit_norm
from seqsfg.config import derive as _audit_derive
from seqsfg.stimulus import render_interval as _audit_render
from seqsfg.measure import frame_rms as _audit_frame_rms
from seqsfg.measure import pair_count_in_lag_range as _audit_pair_count

_AUDIT_FEATURES = (
    "mean_channel_regularity", "max_channel_regularity", "envelope_cv", "rms"
)


def _audit_features(cfg, interval, derived, gain_db=0.0):
    # Every channel is measured, with no access to figure membership. The
    # IEI range is a declared generator parameter, not selected from results.
    regularity = np.asarray([
        _audit_pair_count(
            interval.onset[interval.channel == c].astype(float) * cfg.grid_ms,
            cfg.iei_min_ms, cfg.iei_max_ms, cfg.interval_dur_ms,
        ) for c in range(derived.n_channels)
    ])
    audio = _audit_render(cfg, interval, derived).astype(float)
    audio *= 10.0 ** (float(gain_db) / 20.0)
    envelope = _audit_frame_rms(audio, cfg.sample_rate, frame_ms=2.0)
    return np.asarray([
        regularity.mean(), regularity.max(),
        envelope.std() / max(float(envelope.mean()), 1e-12),
        np.sqrt(np.mean(audio ** 2)),
    ], dtype=float)


def _audit_wilson(k, n, alpha=0.05):
    z = float(_audit_norm.ppf(1.0 - alpha / 2.0))
    p = k / n
    den = 1.0 + z * z / n
    center = (p + z * z / (2.0 * n)) / den
    half = z * np.sqrt(p * (1.0 - p) / n + z * z / (4.0 * n * n)) / den
    return float(center - half), float(center + half)


def _audit_cp(k, n, alpha):
    """Exact binomial interval; alpha is already corrected for multiplicity."""
    lo = 0.0 if k == 0 else float(_audit_beta.ppf(alpha / 2.0, k, n - k + 1))
    hi = 1.0 if k == n else float(_audit_beta.ppf(1.0 - alpha / 2.0, k + 1, n - k))
    return lo, hi


def audit_trials(cfg, trial_factory, n_per_step=24, steps=(0, 14, 23),
                 seed=20260909, planted_gain_db=0.0, n_permutations=1999,
                 alpha=0.05, accuracy_tolerance=0.60):
    """Audit Trial pairs from trial_factory(seed, step).

    Rule: choose the interval with the larger feature score. A fixed choice of
    interval 1 resolves exact ties, with independent randomized target positions.
    Opposite-direction accuracy is also bounded; no direction is fitted to data.

    The global two-sided test uses the maximum absolute standardized mean
    target-minus-foil difference over all feature x step cells. Label flips are
    shared across features within a pair. Step cells have disjoint seeds, and
    no additional pooled row reuses any pair. The denominator is the RMS paired
    difference / sqrt(n), which is invariant under flips. Permutation inference
    assumes pair-label exchangeability under the null. Its positive finding is
    a generator diagnostic, not proof of a perceptual shortcut.

    Wilson intervals are descriptive 95% intervals for individual cells.
    Simultaneous Bonferroni Clopper-Pearson intervals cover all feature x step
    cells at 1-alpha, including either decision direction. Practical tolerance
    requires the entire interval within (1-tolerance, tolerance). A nonsignificant
    p-value by itself is always inconclusive. This bound is conditional on the
    fixed figure/template choices in trial_factory; audit other templates too.

    planted_gain_db=5 adds an artificial target-only 5 dB cue in this audit to
    validate sensitivity. It does not alter notebook listening stimuli. Keep it
    zero for a real diagnostic. Raise n_per_step substantially for useful bounds.
    """
    steps = tuple(float(s) for s in steps)
    n = int(n_per_step)
    B = int(n_permutations)
    if n < 4 or B < 99:
        raise ValueError("Use at least 4 pairs/step and 99 permutations.")
    if not steps or len(set(steps)) != len(steps):
        raise ValueError("steps must be nonempty and unique.")
    if not 0.0 < alpha < 1.0 or not 0.5 < accuracy_tolerance < 1.0:
        raise ValueError("Require 0 < alpha < 1 and 0.5 < tolerance < 1.")
    derived = _audit_derive(cfg)
    rng = np.random.default_rng(int(seed))
    n_features = len(_AUDIT_FEATURES)
    n_cells = len(steps) * n_features
    differences = np.empty((len(steps), n, n_features))
    # Randomized interval order is shared across feature rules for each pair.
    target_first = rng.integers(0, 2, size=(len(steps), n)).astype(bool)
    records = []
    for step_index, step in enumerate(steps):
        for pair_index in range(n):
            trial_seed = int(seed) + 1_000_003 * (1 + step_index * n + pair_index)
            trial = trial_factory(trial_seed, step)
            a = _audit_features(cfg, trial.recurring, derived, planted_gain_db)
            b = _audit_features(cfg, trial.other, derived)
            delta = a - b
            if not np.all(np.isfinite(delta)):
                raise ValueError("Nonfinite feature value: audit aborted.")
            differences[step_index, pair_index] = delta
            records.append({
                "seed": trial_seed, "step_ms": step,
                "target_position": 1 if target_first[step_index, pair_index] else 2,
                "target_scores": dict(zip(_AUDIT_FEATURES, map(float, a))),
                "foil_scores": dict(zip(_AUDIT_FEATURES, map(float, b))),
                "differences": dict(zip(_AUDIT_FEATURES, map(float, delta))),
            })

    # sum(delta) / sqrt(sum(delta**2)) = mean(delta) / (RMS(delta)/sqrt(n)).
    scale = np.sqrt(np.sum(differences ** 2, axis=1))
    safe_scale = np.where(scale > 0, scale, 1.0)
    observed = np.abs(differences.sum(axis=1) / safe_scale)
    null_max = np.empty(B)
    # Small batches avoid allocating B x steps x n x features tensors.
    for begin in range(0, B, 128):
        end = min(B, begin + 128)
        signs = rng.choice([-1.0, 1.0], size=(end - begin, len(steps), n))
        null_statistics = np.abs(np.einsum("bsn,snf->bsf", signs, differences)
                                 / safe_scale[None, :, :])
        null_max[begin:end] = null_statistics.max(axis=(1, 2))
    adjusted_p = (1 + (null_max[:, None, None] >= observed[None, :, :] - 1e-12).sum(axis=0)) / (B + 1)
    global_p = float((1 + np.sum(null_max >= observed.max() - 1e-12)) / (B + 1))
    summary = []
    for step_index, step in enumerate(steps):
        for feature_index, feature in enumerate(_AUDIT_FEATURES):
            delta = differences[step_index, :, feature_index]
            ties = delta == 0.0
            correct = (delta > 0.0) | (ties & target_first[step_index])
            k = int(correct.sum())
            pc = k / n
            wlo, whi = _audit_wilson(k, n)
            clo, chi = _audit_cp(k, n, alpha / n_cells)
            within = bool(clo > 1.0 - accuracy_tolerance and chi < accuracy_tolerance)
            p_adjusted = float(adjusted_p[step_index, feature_index])
            status = ("positive" if p_adjusted < alpha else
                      "within tested tolerance" if within else "inconclusive")
            summary.append({
                "step_ms": step, "feature": feature, "n": n,
                "mean_difference": float(delta.mean()), "ties": int(ties.sum()),
                "pc_target_greater": float(pc),
                "dprime_2afc": float(np.sqrt(2.0) * _audit_norm.ppf((k + 0.5) / (n + 1.0))),
                "wilson_low": wlo, "wilson_high": whi,
                "simultaneous_cp_low": clo, "simultaneous_cp_high": chi,
                "either_direction_accuracy_upper": float(max(chi, 1.0 - clo)),
                "maxT_adjusted_p": p_adjusted,
                "within_tolerance": within, "status": status,
            })
    all_within = all(row["within_tolerance"] for row in summary)
    global_status = ("positive" if global_p < alpha else
                     "within tested tolerance" if all_within else "inconclusive")
    return {
        "records": records, "summary": summary,
        "global": {
            "status": global_status, "paired_maxT_p": global_p,
            "all_within_tolerance": all_within, "n_pairs": n * len(steps),
            "n_feature_step_cells": n_cells, "n_permutations": B,
            "alpha": float(alpha), "accuracy_tolerance": float(accuracy_tolerance),
            "planted_gain_db": float(planted_gain_db), "config_hash": cfg.hash(),
            "seed": int(seed),
            "interpretation": (
                "Positive: at least one prespecified score separates target and foil. "
                "This does not establish that listeners use it. Within tested tolerance: "
                "both score directions are bounded below the selected accuracy tolerance "
                "for these observers and generator settings. Inconclusive: the audit "
                "does not establish a useful bound; nonsignificance is not a clean bill. "
                "Schedule observers have ideal access to per-channel onset times. "
                "Untested observers, templates, and conditions remain unassessed."
            ),
        },
    }

In [ ]:
#@title Live cue challenge — fresh samples, with a detectable-confound option
audit_n=widgets.Dropdown(options=[24,64,128,256,512],value=256,description='Pairs/step')
audit_order=widgets.Dropdown(options=['P','Q'],description='Order')
audit_plant=widgets.Checkbox(value=False,description='Plant +5 dB target cue')
audit_tol=widgets.FloatSlider(value=.60,min=.55,max=.70,step=.01,description='PC bound')
audit_seed=widgets.IntText(value=7701,description='Seed')
audit_button=widgets.Button(description='Run cue audit',button_style='warning')
audit_out=widgets.Output()
LAST_AUDIT=None
def show_audit(result):
    g=result['global']
    print(f"Overall: {g['status']}; global paired-permutation p = {g['paired_maxT_p']:.4f}")
    print(f"{g['n_pairs']} fresh pairs; simultaneous bounds on {g['n_feature_step_cells']} feature × step cells")
    fig,ax=plt.subplots(figsize=(11,4.2))
    for j,row in enumerate(result['summary']):
        ax.errorbar(j,row['pc_target_greater'],
                    yerr=[[row['pc_target_greater']-row['simultaneous_cp_low']],
                          [row['simultaneous_cp_high']-row['pc_target_greater']]],fmt='o',color='#007d8a')
    ax.axhline(.5,color='gray');ax.axhspan(1-g['accuracy_tolerance'],g['accuracy_tolerance'],color='#daece8',alpha=.5)
    ax.set(ylim=(0,1),ylabel='Accuracy: fixed higher-score rule',title='Simultaneous intervals; shaded region is the proposed tolerance')
    ax.set_xticks(range(len(result['summary'])),[f"{r['step_ms']:g} ms\n{r['feature']}" for r in result['summary']],rotation=55,ha='right')
    plt.tight_layout();display(fig);plt.close(fig)
    print(g['interpretation'])
def on_audit(_):
    global LAST_AUDIT
    audit_button.disabled=True
    with audit_out:
        clear_output(wait=True)
        print('Building fresh trials and computing fixed observers. Large samples can take several minutes.')
        try:
            pat=audit_order.value
            LAST_AUDIT=audit_trials(CFG,lambda seed,step:prototype_trial(seed,step,pat),
                                   n_per_step=audit_n.value,steps=[0,14,23],seed=audit_seed.value,
                                   planted_gain_db=5.0 if audit_plant.value else 0.0,
                                   accuracy_tolerance=audit_tol.value)
            LAST_AUDIT['global']['pattern_id']=pat
            show_audit(LAST_AUDIT)
        except Exception as e: print(type(e).__name__+': '+str(e))
    audit_button.disabled=False
audit_button.on_click(on_audit)
display(widgets.VBox([widgets.HBox([audit_n,audit_order,audit_seed]),widgets.HBox([audit_plant,audit_tol]),audit_button,audit_out]))

## 4. Mechanism check: a timing curve cannot identify STDP

The earlier strategy assumed a benefit that vanishes at synchrony and peaks near a plasticity
time constant. The first property is built into the **stimulus**; the second was built into an
assumed curve. They are not evidence for a neural circuit.

Below, the exponential positive-lag kernel peaks at the shortest positive lag, whereas a
phenomenological gamma-shaped curve peaks at its parameter τ. Neither is a prediction for human
accuracy until a specified neural response, learning process, and decision rule connect it to
the waveform. Synaptic timing is not acoustic onset timing ([Bi and Poo, 1998](https://www.gatsby.ucl.ac.uk/~pel/course_wuhan/papers/bi.poo_1998.pdf)).

The printed calculation also checks the earlier rate rule: identical synchronous sustained
activity can change weights symmetrically. **Zero directional asymmetry does not mean zero plasticity.**

In [ ]:
#@title Inspect assumptions — these curves are illustrative, not fitted or observed
def mechanism_sketch(tau_ms=20):
    lag=np.linspace(0,80,401)
    fig,ax=plt.subplots(figsize=(8,3.3))
    ax.plot(lag,np.exp(-lag/tau_ms),label='Exponential positive-lag kernel (limit at 0+)')
    ax.plot(lag,(lag/tau_ms)*np.exp(1-lag/tau_ms),label='Assumed gamma-shaped order benefit')
    ax.set(xlabel='Lag (ms)',ylabel='Arbitrary units',title='A timescale and a curve shape are separate assumptions')
    ax.legend();plt.tight_layout();display(fig);plt.close(fig)
    r=np.zeros((2,30));r[:,10:16]=1
    trace=np.zeros_like(r)
    for k in range(1,21): trace[:,k:]+=np.exp(-k/4)*r[:,:-k]
    update=(r@trace.T-.55*(trace@r.T))/r.shape[1]
    print(f'Synchronous channels: forward update={update[0,1]:.6f}, reverse={update[1,0]:.6f}; asymmetry=0.')
    return update
mechanism_sketch()

## 5. Power the actual learning contrast

This sensitivity analysis simulates the participant endpoint **D = Δtrained − Δuntrained**,
averaged over the positive asynchronies. It includes shared participant ability, order difficulty,
correlated pre/post performance, practice variation, learning-effect heterogeneity, and lapses.
It does not simulate a binding circuit. **Every input effect size is an assumption.**

The provisional full task has 4 pattern pairs × 2 positive steps × 6 trials = **48 trials per
order per phase**, which is the default below. An effect entered as .08 means an eight-percentage-point
extra trained-order gain before the lapse mixture and probability clipping. The plot reports the
actual mean generated contrast. Candidate N values mean **completers**; recruitment allows attrition.
The null run tests false-positive calibration of the same participant-level t test used for power.

This is conditional power, not a recruitment guarantee. Vary plausible effects, baseline accuracy,
heterogeneity and trials. Freeze the smallest worthwhile effect and primary analysis before collecting
confirmatory data. Do not choose the onset step at which pilot improvement happens to be largest.

In [ ]:
#@title Power sensitivity functions
def simulated_contrasts(n_listeners=40,n_trials=48,effect=.08,baseline=.65,
                        ability_sd=.10,effect_sd=.06,lapse=.04,n_sim=500,seed=913):
    rng=np.random.default_rng(seed)
    shape=(n_sim,n_listeners,1)
    ability=rng.normal(0,ability_sd,shape)
    order_bias=rng.normal(0,.025,(n_sim,n_listeners,2))
    baseline_p=baseline+ability+order_bias
    general_gain=rng.normal(.025,.025,shape)
    learning=rng.normal(effect,effect_sd,shape)
    post_p=baseline_p+general_gain+np.concatenate([learning,np.zeros_like(learning)],axis=2)
    pre=(1-lapse)*np.clip(baseline_p,.02,.98)+.5*lapse
    post=(1-lapse)*np.clip(post_p,.02,.98)+.5*lapse
    observed_pre=rng.binomial(n_trials,pre)/n_trials
    observed_post=rng.binomial(n_trials,post)/n_trials
    delta=observed_post-observed_pre
    return delta[:,:,0]-delta[:,:,1], float(np.mean((post-pre)[:,:,0]-(post-pre)[:,:,1]))

def power_grid(effect=.08,n_trials=48,baseline=.65,effect_sd=.06,lapse=.04,attrition=.10,n_sim=500):
    results=[]
    for n in [20,40,60,80,120]:
        contrast,realized=simulated_contrasts(n,n_trials,effect,baseline,effect_sd=effect_sd,lapse=lapse,n_sim=n_sim)
        p=stats.ttest_1samp(contrast,0,axis=1).pvalue
        null,_=simulated_contrasts(n,n_trials,0,baseline,effect_sd=0,lapse=lapse,n_sim=n_sim,seed=1003)
        null_p=stats.ttest_1samp(null,0,axis=1).pvalue
        k=int(np.sum(p<.05));phat=k/n_sim
        lo,hi=_audit_wilson(k,n_sim)
        results.append(dict(completers=n,recruit=int(np.ceil(n/(1-attrition))),power=phat,
                            mc_low=lo,mc_high=hi,null_rejection=float(np.mean(null_p<.05)),
                            realized_mean_contrast=realized,n_sim=n_sim,n_trials_per_order_phase=n_trials))
    return results

def show_power(results):
    fig,ax=plt.subplots(figsize=(8,3.5))
    ax.errorbar([r['completers'] for r in results],[r['power'] for r in results],
                yerr=[[r['power']-r['mc_low'] for r in results],[r['mc_high']-r['power'] for r in results]],fmt='o-',color='#007d8a')
    ax.axhline(.8,ls='--',color='gray');ax.set(xlabel='Completers',ylabel='Conditional power',ylim=(0,1),title='Monte Carlo intervals; assumed effect, not pilot evidence')
    plt.tight_layout();display(fig);plt.close(fig)
    print(' N   recruit   power   null rejection   mean generated D')
    for r in results: print(f"{r['completers']:3d}    {r['recruit']:3d}     {r['power']:.3f}        {r['null_rejection']:.3f}           {r['realized_mean_contrast']:+.3f}")

In [ ]:
#@title Vary sample-size assumptions, then Run simulation
power_effect=widgets.FloatSlider(value=.08,min=0,max=.15,step=.01,description='Extra gain')
power_baseline=widgets.FloatSlider(value=.65,min=.52,max=.85,step=.01,description='Baseline')
power_hetero=widgets.FloatSlider(value=.06,min=0,max=.15,step=.01,description='Effect SD')
power_trials=widgets.Dropdown(options=[24,48,72,96],value=48,description='Trials/cell')
power_lapse=widgets.FloatSlider(value=.04,min=0,max=.20,step=.02,description='Lapse')
power_attrition=widgets.FloatSlider(value=.10,min=0,max=.30,step=.05,description='Attrition')
power_button=widgets.Button(description='Run simulation',button_style='primary')
power_out=widgets.Output()
LAST_POWER=None
def on_power(_):
    global LAST_POWER
    with power_out:
        clear_output(wait=True)
        LAST_POWER=dict(assumptions=dict(effect=power_effect.value,n_trials=power_trials.value,
                        baseline=power_baseline.value,effect_sd=power_hetero.value,lapse=power_lapse.value,
                        attrition=power_attrition.value,n_sim=500))
        LAST_POWER['results']=power_grid(**LAST_POWER['assumptions'])
        show_power(LAST_POWER['results'])
power_button.on_click(on_power)
display(widgets.VBox([widgets.HBox([power_effect,power_baseline,power_hetero]),widgets.HBox([power_trials,power_lapse,power_attrition]),power_button,power_out]))

## 6. Save a reviewable plan

The button creates a ZIP containing the complete plan below, the exact prototype configuration,
source/version provenance, and any live cue audit or power analysis you have run. It never invents
missing results: unrun checks are stored as null. Browser trial CSVs are exported separately by
their panel and can be summarized above.

In [ ]:
#@title Export the full plan and current evidence
FINAL_PLAN = "# Final strategy: learned order and asynchronous figure extraction\n\n**Research question.** Does prior experience with one particular temporal order improve extraction of a recurring auditory figure when its component onsets are separated? The first study estimates a learned-order benefit in a mixture. A grouping mechanism becomes a follow-up hypothesis that needs independent evidence.\n\nThe notebook is a research workbench: hear the stimuli, try an abbreviated pretest–training–posttest, inspect physical controls, and choose a defensible experiment. Its few demonstration trials are not a participant study, a power estimate, or evidence for an effect.\n\n## Why this is a useful direction\n\nPredictive organization already has computational precedents: CHAINS constructs competing perceptual organizations from predictive relationships between events ([Mill et al., 2013](https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1002925)). Temporal-coherence work reports cortical response changes compatible with its proposed mechanism, without establishing all required synaptic operations ([Lu et al., 2025](https://pmc.ncbi.nlm.nih.gov/articles/PMC11903941/)).\n\nAn especially relevant **preprint**, posted April 2026, reports that learned syllable statistics improve target performance more in competition than in quiet, with no comparable benefit from predictable unattended material; the authors favor attentional template matching ([Viswanathan et al., 2026](https://pubmed.ncbi.nlm.nih.gov/42079250/)). SeqSFG can test the contribution of learned temporal order in controlled nonspeech mixtures while explicitly examining that alternative.\n\nThe useful novelty is the combination of a controlled onset-asynchrony manipulation, counterbalanced learning history, and tests that distinguish extraction from recognition. A behavioral benefit alone neither identifies STDP nor establishes pre-attentive processing.\n\n## The causal comparison\n\nCreate two arbitrary orders, P and Q, using exactly the same component frequencies. Each test target repeats one of these orders across its figure elements. Its foil has the same number of grouped elements and the same within-element order, but redraws their frequency sets. The listener chooses the interval in which the same pitches recur.\n\nTrain P in half the listeners and Q in the other half. Both orders appear equally often in the pretest and posttest. “Untrained” therefore means **not specifically trained during acquisition**, not never heard. Training one order exposes every component frequency equally, so familiarity with the component set cannot explain a difference between its two orders.\n\nUse multiple prespecified pattern pairs and counterbalance their assignments across listeners. Generate fresh backgrounds and phases for the posttest. Counterbalance complete test banks across exposure assignments: an identical waveform must be designated trained for some listeners and untrained for others. Do not create acoustically easier targets for the trained label. Independent counterbalancing of bank and phase avoids making a particular bank intrinsically “post.”\n\nFor listener i, define the primary contrast on accuracy:\n\n**Dᵢ = (post trained − pre trained) − (post untrained − pre untrained).**\n\nAverage the four accuracies equally over the prespecified positive-asynchrony conditions, provisionally **14 and 23 ms**, and over pattern pairs. The estimand is the population mean of Dᵢ under this training schedule. A positive value supports order-specific improvement beyond common practice and component familiarity. It remains compatible with sequence-based attention or recognition helping the interval decision.\n\nAt **0 ms**, P and Q are the same physical orderless stimulus. Include one synchrony condition as an anchor; do not manufacture separate trained/untrained observations from identical trials. Absence of an order difference there is stimulus algebra, not a physiological signature.\n\n## A concrete candidate task\n\nFreeze the following after a separate engineering pilot; the timings and counts below are design choices, not established optimal values.\n\n| Stage | Candidate implementation | Purpose |\n|---|---|---|\n| Preparation | Headphones, comfortable fixed level, instructions, 12–20 practice trials with separate material | Establish task understanding without teaching P/Q |\n| Pretest | 4 pattern pairs × 2 orders × 2 positive steps × 6 trials = **96 trials**, plus 16 synchrony trials | Measure each order before dedicated training |\n| Training | **64 feedback trials**, 16 per trained pattern; begin with an easier background and progressively approach test difficulty | Teach the assigned order and target decision |\n| Posttest | **96 fresh trials** balanced exactly like pretest, plus 16 synchrony trials | Estimate the primary change contrast |\n| Learning check | 32 isolated sequence-recognition trials after the primary posttest | Check acquisition without teaching orders before the primary endpoint |\n| Grouping diagnostic | Separate pilot or second session, approximately 64 trials | Validate an independent behavioral consequence of organization |\n\nTarget position is balanced within cells; test order is interleaved, with breaks every 32 trials. Give feedback during practice and training, but not test blocks. Use a fixed training dose; log any fallback in difficulty. Treat this as **instructed perceptual learning**. Training with a clearly audible figure and corrective feedback does not support an implicit-learning claim.\n\nThe first visit contains approximately 320 scored trials including acquisition and the learning check, plus practice: budget around **45–60 minutes**, then replace that estimate with observed pilot completion times. Keep the diagnostic separate if fatigue or repeated exposure would compromise its interpretation. A delayed second visit additionally tests retention, but is a separate endpoint.\n\n## Checks required before recruitment\n\n1. **Physical validity.** Verify per-channel tone budgets, same-channel nonoverlap, permissible element spans, shared target/foil element schedules, RMS, clipping, and reproducibility. Demonstrate that swapping the exposure label changes no test audio.\n2. **Alternative acoustic cues.** Audit the exact final configuration with fresh seeds. The repository's previous pilot showed a single-channel periodicity cue; matched tone counts do not remove temporal cues. Include spectrum, envelope, occupancy, and single-channel onset-history observers.\n3. **Learning alternatives.** Fit observers with the same acquisition exposure as listeners. Compare a learned single-channel timing observer, a temporal-coherence observer, and a learned sequence/template observer. Evaluate held-out target-versus-foil choices; do not grant an ordinary observer the true figure channels or event windows.\n4. **Usable performance range.** An independent pilot of roughly 12–20 listeners should establish comprehension, acquisition, and baseline performance away from floor and ceiling. Adjust the positive steps or background once, then freeze them. Do not choose the primary condition where the pilot's training benefit happens to be largest.\n\nSet a smallest relevant observer advantage before inspecting the final audit, and report uncertainty and held-out performance against that bound. A nonsignificant permutation test alone does not certify cue absence. If a restricted observer succeeds, repair the stimulus or retain a narrower conclusion that acknowledges the available cue.\n\n## A grouping diagnostic and what it can establish\n\nPilot a **component-access task** using the same learned and untrained mixtures. Both intervals contain a recurring figure; one has a small amplitude increment on a randomly selected tone. The listener identifies the interval containing the increment. Cross whether that tone belongs to the figure or the background, and counterbalance its frequency, serial position, and local masking. Balance cue validity and avoid making the increment predictable from training.\n\nFirst establish sensitivity of this diagnostic using clear synchronous grouping versus a disrupted grouping baseline. Adjust increment magnitude independently so easy probe detection does not mask an effect. Then ask whether learned order changes the figure-versus-background access difference, rather than improving all increment judgments equally. This would provide converging evidence about selective access, while still allowing an attention-based explanation. It is not a process-pure binding measure. Objective timing tasks also require validation before being treated as perceptual organization readouts ([Micheyl and Oxenham, 2010](https://pmc.ncbi.nlm.nih.gov/articles/PMC2975891/)).\n\nThe isolated recognition block measures a different outcome. Report its trained-order advantage separately; subtracting its raw accuracy from mixture-detection accuracy would not isolate “binding.” A later experiment can match tasks and baseline difficulty across clear and masked conditions to test a formal learning-by-mixture interaction.\n\n## Analysis, power, and decision rules\n\nPreregister the mean participant contrast D as the single primary endpoint, its two-sided uncertainty interval, and the planned test. A secondary trial-level model may include phase, training assignment, step, pattern identity, trial index, and participant effects; it should corroborate the same estimand rather than replace it after seeing results. Prespecify secondary asynchrony contrasts and multiplicity handling. Do not assume a bell-shaped benefit or interpret an acoustically defined lag as a synaptic time constant.\n\nSimulate the **actual participant contrast**, trial counts, baseline accuracy, participant heterogeneity, within-listener correlations, lapse rates, and attrition. Explore several plausible effect sizes, including a prespecified smallest worthwhile gain, rather than using the largest pilot estimate. Compare candidate samples, for example 40, 60, 80, and 120 completers. Choose the fixed sample and recruitment cap only after that sensitivity analysis; the notebook's illustrative simulation cannot certify power.\n\nExclusions should concern incomplete sessions, playback failure, and prespecified task-comprehension checks. Do not exclude people for lacking a learning benefit or for poor posttest performance. Report the primary analysis for all eligible completers and any acquisition-qualified subgroup only as a prespecified secondary analysis.\n\n**Decision:** proceed to mechanism work if acquisition is demonstrated, the held-out cue audit supports the intended comparison, and the confirmatory order-specific benefit is meaningfully positive. A precise small effect constrains this training regime; failed acquisition calls for redesign, not rejection of all prediction accounts. A positive extraction effect with no independent grouping evidence supports learned sequence-assisted listening. Grouping diagnostics, matched task controls, and competing model predictions determine whether a stronger organization claim is warranted. EEG or circuit-level STDP work comes after that distinction becomes empirically tractable.\n"
import zipfile
EXPORT_DIR=ROOT/'workbench_exports'
export_button=widgets.Button(description='Download plan bundle',button_style='success')
export_out=widgets.Output()
def export_bundle():
    EXPORT_DIR.mkdir(exist_ok=True)
    path=EXPORT_DIR/'SeqSFG_research_plan.zip'
    payload=dict(provenance=provenance(),cue_audit=LAST_AUDIT,power=LAST_POWER,
                 status='Candidate protocol. Requires independent pilot, complete audit, and preregistration.')
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
        z.writestr('research_plan.md',FINAL_PLAN)
        z.writestr('prototype_config.json',json.dumps(CFG.to_dict(),indent=2))
        z.writestr('evidence_and_provenance.json',json.dumps(payload,indent=2))
    return path
def on_export(_):
    with export_out:
        clear_output(wait=True)
        path=export_bundle()
        if IN_COLAB:
            from google.colab import files
            files.download(str(path))
        else: display(FileLink(str(path)))
        print('Saved full candidate plan, configuration, and available checks. Trial CSVs export from the trial panel.')
export_button.on_click(on_export)
display(widgets.VBox([export_button,export_out]))

# Final strategy: learned order and asynchronous figure extraction

**Research question.** Does prior experience with one particular temporal order improve extraction of a recurring auditory figure when its component onsets are separated? The first study estimates a learned-order benefit in a mixture. A grouping mechanism becomes a follow-up hypothesis that needs independent evidence.

The notebook is a research workbench: hear the stimuli, try an abbreviated pretest–training–posttest, inspect physical controls, and choose a defensible experiment. Its few demonstration trials are not a participant study, a power estimate, or evidence for an effect.

## Why this is a useful direction

Predictive organization already has computational precedents: CHAINS constructs competing perceptual organizations from predictive relationships between events ([Mill et al., 2013](https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1002925)). Temporal-coherence work reports cortical response changes compatible with its proposed mechanism, without establishing all required synaptic operations ([Lu et al., 2025](https://pmc.ncbi.nlm.nih.gov/articles/PMC11903941/)).

An especially relevant **preprint**, posted April 2026, reports that learned syllable statistics improve target performance more in competition than in quiet, with no comparable benefit from predictable unattended material; the authors favor attentional template matching ([Viswanathan et al., 2026](https://pubmed.ncbi.nlm.nih.gov/42079250/)). SeqSFG can test the contribution of learned temporal order in controlled nonspeech mixtures while explicitly examining that alternative.

The useful novelty is the combination of a controlled onset-asynchrony manipulation, counterbalanced learning history, and tests that distinguish extraction from recognition. A behavioral benefit alone neither identifies STDP nor establishes pre-attentive processing.

## The causal comparison

Create two arbitrary orders, P and Q, using exactly the same component frequencies. Each test target repeats one of these orders across its figure elements. Its foil has the same number of grouped elements and the same within-element order, but redraws their frequency sets. The listener chooses the interval in which the same pitches recur.

Train P in half the listeners and Q in the other half. Both orders appear equally often in the pretest and posttest. “Untrained” therefore means **not specifically trained during acquisition**, not never heard. Training one order exposes every component frequency equally, so familiarity with the component set cannot explain a difference between its two orders.

Use multiple prespecified pattern pairs and counterbalance their assignments across listeners. Generate fresh backgrounds and phases for the posttest. Counterbalance complete test banks across exposure assignments: an identical waveform must be designated trained for some listeners and untrained for others. Do not create acoustically easier targets for the trained label. Independent counterbalancing of bank and phase avoids making a particular bank intrinsically “post.”

For listener i, define the primary contrast on accuracy:

**Dᵢ = (post trained − pre trained) − (post untrained − pre untrained).**

Average the four accuracies equally over the prespecified positive-asynchrony conditions, provisionally **14 and 23 ms**, and over pattern pairs. The estimand is the population mean of Dᵢ under this training schedule. A positive value supports order-specific improvement beyond common practice and component familiarity. It remains compatible with sequence-based attention or recognition helping the interval decision.

At **0 ms**, P and Q are the same physical orderless stimulus. Include one synchrony condition as an anchor; do not manufacture separate trained/untrained observations from identical trials. Absence of an order difference there is stimulus algebra, not a physiological signature.

## A concrete candidate task

Freeze the following after a separate engineering pilot; the timings and counts below are design choices, not established optimal values.

| Stage | Candidate implementation | Purpose |
|---|---|---|
| Preparation | Headphones, comfortable fixed level, instructions, 12–20 practice trials with separate material | Establish task understanding without teaching P/Q |
| Pretest | 4 pattern pairs × 2 orders × 2 positive steps × 6 trials = **96 trials**, plus 16 synchrony trials | Measure each order before dedicated training |
| Training | **64 feedback trials**, 16 per trained pattern; begin with an easier background and progressively approach test difficulty | Teach the assigned order and target decision |
| Posttest | **96 fresh trials** balanced exactly like pretest, plus 16 synchrony trials | Estimate the primary change contrast |
| Learning check | 32 isolated sequence-recognition trials after the primary posttest | Check acquisition without teaching orders before the primary endpoint |
| Grouping diagnostic | Separate pilot or second session, approximately 64 trials | Validate an independent behavioral consequence of organization |

Target position is balanced within cells; test order is interleaved, with breaks every 32 trials. Give feedback during practice and training, but not test blocks. Use a fixed training dose; log any fallback in difficulty. Treat this as **instructed perceptual learning**. Training with a clearly audible figure and corrective feedback does not support an implicit-learning claim.

The first visit contains approximately 320 scored trials including acquisition and the learning check, plus practice: budget around **45–60 minutes**, then replace that estimate with observed pilot completion times. Keep the diagnostic separate if fatigue or repeated exposure would compromise its interpretation. A delayed second visit additionally tests retention, but is a separate endpoint.

## Checks required before recruitment

1. **Physical validity.** Verify per-channel tone budgets, same-channel nonoverlap, permissible element spans, shared target/foil element schedules, RMS, clipping, and reproducibility. Demonstrate that swapping the exposure label changes no test audio.
2. **Alternative acoustic cues.** Audit the exact final configuration with fresh seeds. The repository's previous pilot showed a single-channel periodicity cue; matched tone counts do not remove temporal cues. Include spectrum, envelope, occupancy, and single-channel onset-history observers.
3. **Learning alternatives.** Fit observers with the same acquisition exposure as listeners. Compare a learned single-channel timing observer, a temporal-coherence observer, and a learned sequence/template observer. Evaluate held-out target-versus-foil choices; do not grant an ordinary observer the true figure channels or event windows.
4. **Usable performance range.** An independent pilot of roughly 12–20 listeners should establish comprehension, acquisition, and baseline performance away from floor and ceiling. Adjust the positive steps or background once, then freeze them. Do not choose the primary condition where the pilot's training benefit happens to be largest.

Set a smallest relevant observer advantage before inspecting the final audit, and report uncertainty and held-out performance against that bound. A nonsignificant permutation test alone does not certify cue absence. If a restricted observer succeeds, repair the stimulus or retain a narrower conclusion that acknowledges the available cue.

## A grouping diagnostic and what it can establish

Pilot a **component-access task** using the same learned and untrained mixtures. Both intervals contain a recurring figure; one has a small amplitude increment on a randomly selected tone. The listener identifies the interval containing the increment. Cross whether that tone belongs to the figure or the background, and counterbalance its frequency, serial position, and local masking. Balance cue validity and avoid making the increment predictable from training.

First establish sensitivity of this diagnostic using clear synchronous grouping versus a disrupted grouping baseline. Adjust increment magnitude independently so easy probe detection does not mask an effect. Then ask whether learned order changes the figure-versus-background access difference, rather than improving all increment judgments equally. This would provide converging evidence about selective access, while still allowing an attention-based explanation. It is not a process-pure binding measure. Objective timing tasks also require validation before being treated as perceptual organization readouts ([Micheyl and Oxenham, 2010](https://pmc.ncbi.nlm.nih.gov/articles/PMC2975891/)).

The isolated recognition block measures a different outcome. Report its trained-order advantage separately; subtracting its raw accuracy from mixture-detection accuracy would not isolate “binding.” A later experiment can match tasks and baseline difficulty across clear and masked conditions to test a formal learning-by-mixture interaction.

## Analysis, power, and decision rules

Preregister the mean participant contrast D as the single primary endpoint, its two-sided uncertainty interval, and the planned test. A secondary trial-level model may include phase, training assignment, step, pattern identity, trial index, and participant effects; it should corroborate the same estimand rather than replace it after seeing results. Prespecify secondary asynchrony contrasts and multiplicity handling. Do not assume a bell-shaped benefit or interpret an acoustically defined lag as a synaptic time constant.

Simulate the **actual participant contrast**, trial counts, baseline accuracy, participant heterogeneity, within-listener correlations, lapse rates, and attrition. Explore several plausible effect sizes, including a prespecified smallest worthwhile gain, rather than using the largest pilot estimate. Compare candidate samples, for example 40, 60, 80, and 120 completers. Choose the fixed sample and recruitment cap only after that sensitivity analysis; the notebook's illustrative simulation cannot certify power.

Exclusions should concern incomplete sessions, playback failure, and prespecified task-comprehension checks. Do not exclude people for lacking a learning benefit or for poor posttest performance. Report the primary analysis for all eligible completers and any acquisition-qualified subgroup only as a prespecified secondary analysis.

**Decision:** proceed to mechanism work if acquisition is demonstrated, the held-out cue audit supports the intended comparison, and the confirmatory order-specific benefit is meaningfully positive. A precise small effect constrains this training regime; failed acquisition calls for redesign, not rejection of all prediction accounts. A positive extraction effect with no independent grouping evidence supports learned sequence-assisted listening. Grouping diagnostics, matched task controls, and competing model predictions determine whether a stronger organization claim is warranted. EEG or circuit-level STDP work comes after that distinction becomes empirically tractable.